In [ ]:
!pip install -q transformers accelerate bitsandbytes huggingface_hub

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from huggingface_hub import login

login()  # paste your HF token

model_id = "google/gemma-3-4b-it"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id, torch_dtype=torch.bfloat16, device_map="auto"
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.7 MB/s eta 0:00:00


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

...

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/90.6k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

#SINHALA LANGUAGE

In [ ]:
import json

DATA_PATH = "/content/sample_data/sri_lankan_simplified.json"

with open(DATA_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"Loaded {len(data)} items")

Loaded 203 items


In [ ]:
INSTRUCTION_PROMPT = (
    """ You are a Sinhala Expert for answering multiple-choice questions.
    You need to always evaluate these questions based on the Sri Lankan context and provide the most accurate answer.
    You should only respond with the correct option text, without any additional explanation or commentary.

Question: {question}
Option A: {option A}
Option B: {option B}
Option C: පිළිතුරු දෙකම නිවැරදියි.
Option D: පිළිතුරු දෙකම නිවැරදි නොවේ.
Return only the correct option text. Expected output is either the text of 'A', 'B', 'C', or 'D'.
"""
)

def build_prompt(item):
    lines = [INSTRUCTION_PROMPT, "", item["scenario_question"], "", "Options:"]
    for letter, text in item["options"].items():
        lines.append(f"{letter}. {text}")
    return "\n".join(lines)

In [ ]:
import re
def generate_answer(prompt, valid_letters, max_new_tokens=20):
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True,
    ).to(model.device)

    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)

    input_len = inputs["input_ids"].shape[-1]
    raw = tokenizer.decode(output[0][input_len:], skip_special_tokens=True).strip()

    # First try a clean letter match (model followed instructions)
    match = re.search(r"\b([A-D])\b", raw.upper())
    if match and match.group(1) in valid_letters:
        predicted = match.group(1)
    # Fall back: model echoed the option content instead of the letter
    elif "දෙකම නිවැරදි නොවේ" in raw:
        predicted = "0"
    elif "දෙකම නිවැරදි" in raw:   # check this AFTER the "නොවේ" (negative) case
        predicted = "Both"
    else:
        predicted = None

    return predicted, raw

In [ ]:
OUTPUT_PATH = "/content/sri_lankan_results.json"

for i, item in enumerate(data):
    if item.get("model_answer"):  # already processed, skip (resume-safe)
        continue

    prompt = build_prompt(item)
    predicted, raw_output = generate_answer(prompt, list(item["options"].keys()))

    item["model_answer"] = predicted
    item["raw_model_output"] = raw_output

    with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

    print(f"[{i+1}/{len(data)}] {item['id']} -> {predicted} (gold: {item['gold_answer']})")

print("All done.")

[1/203] SI_2004 -> None (gold: A)
[2/203] SI_2007 -> B (gold: B)
[3/203] SI_2009 -> Both (gold: A)
[4/203] SI_2012 -> None (gold: B)
[5/203] SI_2015 -> A (gold: A)
[6/203] SI_2017 -> A (gold: A)
[7/203] SI_2018 -> A (gold: A)
[8/203] SI_2022 -> 0 (gold: A)
[9/203] SI_2026 -> A (gold: B)
[10/203] SI_2027 -> A (gold: A)
[11/203] SI_2028 -> A (gold: A)
[12/203] SI_2033 -> B (gold: B)
[13/203] SI_2036 -> A (gold: A)
[14/203] SI_2038 -> A (gold: A)
[15/203] SI_2039 -> Both (gold: A)
[16/203] SI_2046 -> Both (gold: A)
[17/203] SI_2049 -> 0 (gold: B)
[18/203] SI_2056 -> A (gold: A)
[19/203] SI_2092 -> A (gold: A)
[20/203] SI_2097 -> A (gold: A)
[21/203] SI_2100 -> A (gold: Both)
[22/203] SI_2105 -> B (gold: A)
[23/203] SI_2111 -> A (gold: A)
[24/203] SI_2112 -> A (gold: A)
[25/203] SI_2124 -> A (gold: A)
[26/203] SI_2127 -> A (gold: A)
[27/203] SI_2129 -> A (gold: A)
[28/203] SI_2132 -> B (gold: B)
[29/203] SI_2151 -> A (gold: A)
[30/203] SI_2157 -> A (gold: A)
[31/203] SI_2158 -> A (gold: A)

In [ ]:
import json

with open("/content/sri_lankan_results.json", "r", encoding="utf-8") as f:
    results = json.load(f)

correct = sum(1 for item in results if item.get("model_answer") == item.get("gold_answer"))
total = len(results)

print(f"Correct: {correct} / {total}")
print(f"Accuracy: {correct / total:.2%}")

Correct: 128 / 203
Accuracy: 63.05%


#CHINESE LANGUAGE

In [ ]:
import json

DATA_PATH = "/content/sample_data/chinese_simplified.json"

with open(DATA_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"Loaded {len(data)} items")

Loaded 790 items


In [ ]:
INSTRUCTION_PROMPT = (
    """
You are a Chinese Expert for answering multiple-choice questions.
You need to always evaluate these questions based on the Chinese context and provide the most accurate answer.
You should only respond with the correct option text, without any additional explanation or commentary.

Question: {question}
Option A: {option A}
Option B: {option B}
Option C: {option C}
Option D: {option D}
Return only the correct option text. Expected output is either the text of 'A', 'B', 'C', or 'D'. "Question": "

"""
)

def build_prompt(item):
    lines = [INSTRUCTION_PROMPT, "", item["scenario_question"], "", "Options:"]
    for letter, text in item["options"].items():
        lines.append(f"{letter}. {text}")
    return "\n".join(lines)

In [ ]:
import re
def generate_answer(prompt, valid_letters, max_new_tokens=20):
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True,
    ).to(model.device)

    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)

    input_len = inputs["input_ids"].shape[-1]
    raw = tokenizer.decode(output[0][input_len:], skip_special_tokens=True).strip()

    # First try a clean letter match (model followed instructions)
    match = re.search(r"\b([A-D])\b", raw.upper())
    if match and match.group(1) in valid_letters:
        predicted = match.group(1)
    else:
        predicted = None

    return predicted, raw

In [ ]:
OUTPUT_PATH = "/content/chinese_results.json"

for i, item in enumerate(data):
    if item.get("model_answer"):  # already processed, skip (resume-safe)
        continue

    prompt = build_prompt(item)
    predicted, raw_output = generate_answer(prompt, list(item["options"].keys()))

    item["model_answer"] = predicted
    item["raw_model_output"] = raw_output

    with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

    print(f"[{i+1}/{len(data)}] {item['id']} -> {predicted} (gold: {item['gold_answer']})")

print("All done.")

[1/790] ZH_0_0 -> A (gold: A)
[2/790] ZH_0_3 -> D (gold: C)
[3/790] ZH_0_41 -> A (gold: A)
[4/790] ZH_0_42 -> B (gold: B)
[5/790] ZH_0_64 -> A (gold: A)
[6/790] ZH_1_47 -> C (gold: A)
[7/790] ZH_1_57 -> D (gold: A)
[8/790] ZH_1_58 -> A (gold: A)
[9/790] ZH_1_69 -> C (gold: A)
[10/790] ZH_1_77 -> B (gold: A)
[11/790] ZH_2_27 -> A (gold: A)
[12/790] ZH_2_48 -> A (gold: A)
[13/790] ZH_2_57 -> A (gold: A)
[14/790] ZH_2_60 -> C (gold: B)
[15/790] ZH_2_69 -> D (gold: A)
[16/790] ZH_3_14 -> A (gold: C)
[17/790] ZH_3_15 -> C (gold: A)
[18/790] ZH_3_16 -> D (gold: A)
[19/790] ZH_3_22 -> D (gold: C)
[20/790] ZH_3_79 -> A (gold: B)
[21/790] ZH_4_9 -> B (gold: B)
[22/790] ZH_4_23 -> B (gold: C)
[23/790] ZH_4_38 -> A (gold: D)
[24/790] ZH_4_46 -> B (gold: B)
[25/790] ZH_4_74 -> A (gold: A)
[26/790] ZH_5_14 -> A (gold: A)
[27/790] ZH_5_22 -> B (gold: A)
[28/790] ZH_5_73 -> A (gold: A)
[29/790] ZH_5_75 -> D (gold: D)
[30/790] ZH_5_76 -> A (gold: A)
[31/790] ZH_6_20 -> A (gold: A)
[32/790] ZH_6_46 -> 

In [ ]:
import json

with open("/content/chinese_results.json", "r", encoding="utf-8") as f:
    results = json.load(f)

correct = sum(1 for item in results if item.get("model_answer") == item.get("gold_answer"))
total = len(results)

print(f"Correct: {correct} / {total}")
print(f"Accuracy: {correct / total:.2%}")

Correct: 333 / 790
Accuracy: 42.15%


#INDONESIAN LANGUAGE

In [ ]:
import json

DATA_PATH = "/content/sample_data/indonesian_simplified.json"

with open(DATA_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"Loaded {len(data)} items")

Loaded 366 items


In [ ]:
INSTRUCTION_PROMPT = (
    """
    You are an Indonesian Expert for answering multiple-choice questions.
    You need to always evaluate these questions based on the Indonesian context and provide the most accurate answer.
    You should only respond with the correct option text, without any additional explanation or commentary.

Question: {question}
Option A: {option A}
Option B: {option B}
Option C: {option C}
Option D: {option D}
Return only the correct option text. Expected output is either the text of 'A', 'B', 'C', or 'D'.


"""
)

def build_prompt(item):
    lines = [INSTRUCTION_PROMPT, "", item["scenario_question"], "", "Options:"]
    for letter, text in item["options"].items():
        lines.append(f"{letter}. {text}")
    return "\n".join(lines)

In [ ]:
import re
def generate_answer(prompt, valid_letters, max_new_tokens=20):
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True,
    ).to(model.device)

    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)

    input_len = inputs["input_ids"].shape[-1]
    raw = tokenizer.decode(output[0][input_len:], skip_special_tokens=True).strip()

    # First try a clean letter match (model followed instructions)
    match = re.search(r"\b([A-D])\b", raw.upper())
    if match and match.group(1) in valid_letters:
        predicted = match.group(1)
    else:
        predicted = None

    return predicted, raw

In [ ]:
OUTPUT_PATH = "/content/indonesian_results.json"

for i, item in enumerate(data):
    if item.get("model_answer"):  # already processed, skip (resume-safe)
        continue

    prompt = build_prompt(item)
    predicted, raw_output = generate_answer(prompt, list(item["options"].keys()))

    item["model_answer"] = predicted
    item["raw_model_output"] = raw_output

    with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

    print(f"[{i+1}/{len(data)}] {item['id']} -> {predicted} (gold: {item['gold_answer']})")

print("All done.")

[1/366] ID_3 -> C (gold: A)
[2/366] ID_5 -> B (gold: B)
[3/366] ID_6 -> C (gold: D)
[4/366] ID_9 -> B (gold: A)
[5/366] ID_15 -> A (gold: A)
[6/366] ID_17 -> A (gold: A)
[7/366] ID_24 -> D (gold: D)
[8/366] ID_28 -> D (gold: A, D)
[9/366] ID_36 -> B (gold: B)
[10/366] ID_38 -> A (gold: C)
[11/366] ID_42 -> C (gold: C)
[12/366] ID_47 -> B (gold: C, D)
[13/366] ID_52 -> C (gold: B)
[14/366] ID_58 -> A (gold: A)
[15/366] ID_61 -> A (gold: A)
[16/366] ID_74 -> A (gold: A)
[17/366] ID_91 -> A (gold: A)
[18/366] ID_97 -> B (gold: C)
[19/366] ID_98 -> A (gold: A)
[20/366] ID_102 -> C (gold: C)
[21/366] ID_108 -> A (gold: A, D)
[22/366] ID_110 -> B (gold: B)
[23/366] ID_115 -> C (gold: C)
[24/366] ID_117 -> D (gold: D)
[25/366] ID_133 -> B (gold: A)
[26/366] ID_137 -> C (gold: C)
[27/366] ID_146 -> C (gold: C)
[28/366] ID_148 -> B (gold: D)
[29/366] ID_149 -> A (gold: A)
[30/366] ID_152 -> D (gold: B, D)
[31/366] ID_155 -> C (gold: D)
[32/366] ID_156 -> C (gold: C, D)
[33/366] ID_158 -> C (gol

In [ ]:
import json

with open("/content/indonesian_results.json", "r", encoding="utf-8") as f:
    results = json.load(f)

correct = 0
for item in results:
    model_answer = item.get("model_answer")
    gold_answer = item.get("gold_answer", "")

    gold_letters = [g.strip() for g in gold_answer.split(",")]  # handles single or multiple gold answers

    if model_answer is not None and model_answer.strip() in gold_letters:
        correct += 1

total = len(results)

print(f"Correct: {correct} / {total}")
print(f"Accuracy: {correct / total:.2%}")

Correct: 235 / 366
Accuracy: 64.21%
